In [11]:
"""
mo_export_tf_activity_matrix.ipynb

This script exports the inferred TF activity matrix from a Multiome SCENIC+ analysis to mtx and tsv formats

authors: Roy Oelen

"""

'\nmo_export_tf_activity_matrix.ipynb\n\nThis script exports the inferred TF activity matrix from a Multiome SCENIC+ analysis to mtx and tsv formats\n\nauthors: Roy Oelen\n\n'

In [9]:
###########
# imports #
###########

# for path operations
import os
import glob
from pathlib import Path
# for reading the output of SCENIC+
import scanpy as sc
import anndata
import mudata
from scenicplus.RSS import (regulon_specificity_scores, plot_rss)
# for saving results
import joblib
# for visualizing results
import pandas as pd
# for making a checksum
import hashlib
# convert to matrix
from scipy.sparse import csr_array
# write zipped mtx file
import gzip
# to write sparse matrices
import scipy.io


In [ ]:
#############
# functions #
#############

# method to create md5
def create_md5_file(input_file):
    """
    Creates an MD5 hash of the specified file and writes it to a new file with the same name but .md5 added to the extension.

    Args:
        input_file (str): The path to the input file for which the MD5 hash should be created.

    Returns:
        int: Returns 0 on success, 1 on failure.

    Raises:
        FileNotFoundError: Thrown if the input file does not exist.
        IOError: Thrown if there is an error reading the input file (like permission denied) or writing the output md5 file.
    """
    try:
        # get an md5 of the file
        digest = None
        with open(input_file, "rb") as f:
            # try Python 3.11+ method if it is available
            if callable(getattr(hashlib, 'file_digest', None)):
                # digest with one command
                digest = hashlib.file_digest(f, 'md5')
            # or the older 3.8+ method if we don't have the newer method
            else:
                # initialize digest
                digest = hashlib.md5()
                # read file in chunks
                while chunk := f.read(8192):
                    # update digestion
                    digest.update(chunk)       
        # get the output path of the md5
        output_md5_loc = ''.join([input_file, '.md5'])
        # and write that
        with open(output_md5_loc, "w") as m:
            m.write(digest.hexdigest())
        # upon success, return 0
        return 0
    except Exception as e:
        print(f"Exception occured upon md5 file creation: {e}")
        return 1



In [4]:
###################
# paths of inputs #
###################

# location of the mu that has the eregulon data
eregulon_mu_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/scplusmdata.h5mu'


In [5]:
###################
# load the inputs #
###################

# read the scenic output metadata
scplus_mdata = mudata.read(eregulon_mu_loc)


/home/umcg-roelen/miniconda3/envs/pycistopic_env/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/umcg-roelen/miniconda3/envs/pycistopic_env/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/umcg-roelen/miniconda3/envs/pycistopic_env/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/umcg-roelen/miniconda3/envs/pycistopic_env/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/umcg-roelen/miniconda3/envs/pycistopic_env/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be remo

In [6]:
#######################################
# process eRegulons data based on AUC #
#######################################

# merge the direct and extended
eRegulon_gene_AUC = anndata.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)
# add the cell names
eRegulon_gene_AUC.obs = scplus_mdata.obs.loc[eRegulon_gene_AUC.obs_names]

In [7]:
# create a mapping dictionary for the major cell types to the lineages
lowres_to_lineage = {
    'monocyte' : 'myeloid',
    'DC' : 'myeloid', 
    'B' : 'lymphoid',
    'plasmablast' : 'lymphoid',
    'CD4T' : 'lymphoid', 
    'CD8T' : 'lymphoid', 
    'NK' : 'lymphoid', 
    'unannotated' : 'unannotated',
    'T_other' : 'lymphoid'
}
# and add this gene auc object
scplus_mdata.obs["lineage"] = scplus_mdata.obs["scATAC_counts:cell_type"].map(lowres_to_lineage)


In [8]:
# create a mapping dictionary for the major cell types a merged one
lowres_to_lowerres = {
    'monocyte' : 'monocyte',
    'DC' : 'DC', 
    'B' : 'B',
    'plasmablast' : 'B',
    'CD4T' : 'CD4T', 
    'CD8T' : 'CD8T', 
    'NK' : 'NK', 
    'unannotated' : 'unannotated',
    'T_other' : 'T_other'
}
# and add this gene auc object
scplus_mdata.obs["celltype_lowerres_merged"] = scplus_mdata.obs["scATAC_counts:cell_type"].map(lowres_to_lowerres)


In [26]:
###################
# save new object #
###################

# location to store the object
eRegulon_gene_AUC_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/eregulon_gene_auc.joblib'
# store the actual object
joblib.dump(eRegulon_gene_AUC, eRegulon_gene_AUC_loc)
# and make a checksum
create_md5_file(eRegulon_gene_AUC_loc)


0

In [25]:
##################
# save as matrix #
##################

# location where object was stored
eRegulon_gene_AUC_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/eregulon_gene_auc.joblib'
# load object
eRegulon_gene_AUC = joblib.load(eRegulon_gene_AUC_loc)

# convert to a sparse matrix
eRegulon_gene_AUC_csr = csr_array(eRegulon_gene_AUC.X).tocsr().transpose()
# extract the cell names
cell_names = eRegulon_gene_AUC.obs_names.tolist()
# extract the eRegulon names
eregulon_names = eRegulon_gene_AUC.var_names.tolist()

# set the output locations
eRegulon_gene_AUC_mtx_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/eregulon_gene_auc.mtx.gz'
eRegulon_gene_AUC_eregulon_names_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/eregulon_gene_auc_eregnames.txt.gz'
eRegulon_gene_AUC_cell_names_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/eregulon_gene_auc_barcodes.txt.gz'

# write these files
pd.DataFrame(data = cell_names).to_csv(path_or_buf = eRegulon_gene_AUC_cell_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')
pd.DataFrame(data = eregulon_names).to_csv(path_or_buf = eRegulon_gene_AUC_eregulon_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')
with gzip.open(eRegulon_gene_AUC_mtx_loc, 'wb') as f:
    scipy.io.mmwrite(f, eRegulon_gene_AUC_csr)

# and make checksums
create_md5_file(eRegulon_gene_AUC_mtx_loc)
create_md5_file(eRegulon_gene_AUC_eregulon_names_loc)
create_md5_file(eRegulon_gene_AUC_cell_names_loc)


0

In [10]:
# go through each cell type in the major cell types and save a matrix for each of those too
for celltype in scplus_mdata.obs['scATAC_counts:cell_type'].unique() :
    # extract the cells of this cell type
    cells_of_type = scplus_mdata.obs_names[scplus_mdata.obs['scATAC_counts:cell_type'] == celltype]
    # subset the eRegulon gene AUC object
    eRegulon_gene_AUC_subset = eRegulon_gene_AUC[cells_of_type, :]
    # convert to table with first row the cell barcodes and first column the eRegulon names
    eRegulon_gene_AUC_subset_csr = csr_array(eRegulon_gene_AUC_subset.X).tocsr().transpose()
    # extract the cell names
    cell_names_subset = eRegulon_gene_AUC_subset.obs_names.tolist()
    # extract the eRegulon names
    eregulon_names_subset = eRegulon_gene_AUC_subset.var_names.tolist()
    # set the output locations
    eRegulon_gene_AUC_subset_mtx_loc = f'/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/eregulon_gene_auc_{celltype}.mtx.gz'
    eRegulon_gene_AUC_subset_eregulon_names_loc = f'/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/eregulon_gene_auc_{celltype}_eregnames.txt.gz'
    eRegulon_gene_AUC_subset_cell_names_loc = f'/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/eregulon_gene_auc_{celltype}_barcodes.txt.gz'
    # write these files
    pd.DataFrame(data = cell_names_subset).to_csv(path_or_buf = eRegulon_gene_AUC_subset_cell_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')
    pd.DataFrame(data = eregulon_names_subset).to_csv(path_or_buf = eRegulon_gene_AUC_subset_eregulon_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')
    with gzip.open(eRegulon_gene_AUC_subset_mtx_loc, 'wb') as f:
        scipy.io.mmwrite(f, eRegulon_gene_AUC_subset_csr)
    # and make checksums
    create_md5_file(eRegulon_gene_AUC_subset_mtx_loc)
    create_md5_file(eRegulon_gene_AUC_subset_eregulon_names_loc)
    create_md5_file(eRegulon_gene_AUC_subset_cell_names_loc)

    # now also make a non-sparse version for easier loading in R
    eRegulon_gene_AUC_subset_mtx_loc_nonsparse = f'/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/scplus_pipeline_merged_major_and_minor_celltypes/output/eregulon_gene_auc_{celltype}_nonsparse.tsv.gz'
    # write non-sparse file
    pd.DataFrame(data = eRegulon_gene_AUC_subset.X, index = cell_names_subset, columns = eregulon_names_subset).to_csv(path_or_buf = eRegulon_gene_AUC_subset_mtx_loc_nonsparse, sep = '\t', header = True, index = True, compression = 'gzip')
    # and make checksum
    create_md5_file(eRegulon_gene_AUC_subset_mtx_loc_nonsparse)
